<a href="https://colab.research.google.com/github/TiffanySGee/Jasmine-Gee-grade8-library-Repo/blob/main/Jasmine_Gee_grade8_library_Repo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import requests
import json
import firebase_admin
from firebase_admin import credentials, db

# --- 1. Verify JSON Upload from GitHub ---
github_raw_url = 'https://raw.githubusercontent.com/TiffanySGee/Jasmine-Gee-grade8-library-Repo/refs/heads/main/books.json'
response = requests.get(github_raw_url)

if response.status_code == 200:
    books_data = response.json()
    print("✅ 1. GitHub JSON accessed successfully!")
else:
    print(f"❌ 1. Error: Could not find GitHub file. Status Code: {response.status_code}")
    # Initialize books_data to an empty dictionary to prevent further errors
    books_data = {}

# --- 2. Connect Python to Firebase ---
# IMPORTANT: Replace the URL below with YOUR actual Firebase Database URL
firebase_url = 'https://jasmine-gee-library-repo-default-rtdb.firebaseio.com/'

# If a Firebase app has already been initialized, delete it to allow re-initialization with new config
if firebase_admin._apps:
    firebase_admin.delete_app(firebase_admin.get_app())

cred = credentials.Certificate('firebase_key.json')
firebase_admin.initialize_app(cred, {
    'databaseURL': firebase_url
})
print("✅ 2. Firebase connection established!")

# --- 3. Upload books.json data to database ---
# Only attempt to upload if books_data was successfully retrieved and is not empty
if books_data:
    ref = db.reference('/') # This points to the root of your database
    ref.set(books_data)
    print("✅ 3. Data successfully uploaded to Firebase!")
else:
    print("⚠️ 3. Data not uploaded to Firebase as GitHub JSON was not successfully accessed or was empty.")

# --- 4. DISPLAY all the books data ---
db_data = ref.get()
print("\n--- 4. DATA FROM DATABASE ---")
print(json.dumps(db_data, indent=2))

# --- 5, 6, 7. Specific Displays and Calculations ---
# Add checks to ensure 'books' key exists before iterating
if 'books' in db_data and db_data['books']:
    print("\n--- 5. BOOK TITLES ---")
    for book in db_data['books']:
        print(f"Title: {book['title']}")

    print("\n--- 6. AVAILABILITY ---")
    for book in db_data['books']:
        # Add a check for 'details' and 'available' keys
        availability = book.get('details', {}).get('available', 'N/A')
        print(f"{book['title']} | Available: {availability}")

    total_copies = sum(book.get('details', {}).get('copies', 0) for book in db_data['books'])
    print(f"\n✅ 7. TOTAL BOOKS IN LIBRARY: {total_copies}")
else:
    print("\n--- 5, 6, 7. No book data found in the database for display and calculation.")

✅ 1. GitHub JSON accessed successfully!
✅ 2. Firebase connection established!
✅ 3. Data successfully uploaded to Firebase!

--- 4. DATA FROM DATABASE ---
{
  "books": [
    {
      "author": "Juan Dela Cruz",
      "details": {
        "available": 3,
        "copies": 5
      },
      "id": 2001,
      "title": "Introduction to Python"
    },
    {
      "author": "Maria Santos",
      "details": {
        "available": 2,
        "copies": 4
      },
      "id": 2002,
      "title": "Data Structures Basics"
    },
    {
      "author": "Carlos Reyes",
      "details": {
        "available": 6,
        "copies": 6
      },
      "id": 2003,
      "title": "Web Development Essentials"
    }
  ]
}

--- 5. BOOK TITLES ---
Title: Introduction to Python
Title: Data Structures Basics
Title: Web Development Essentials

--- 6. AVAILABILITY ---
Introduction to Python | Available: 3
Data Structures Basics | Available: 2
Web Development Essentials | Available: 6

✅ 7. TOTAL BOOKS IN LIBRARY: 15
